# Cucker-Smale Benchmark

This notebook is the compact validation page for flocking control.

## Environment Basics

- State space: $(x_t,v_t)\in\mathbb{R}^2$, position and velocity.
- Action space: $u_t\in\mathbb{R}$, acceleration/control.
- Population law: empirical particle law over positions and velocities.
- Goal: reduce velocity dispersion and spatial spread while controlling energy.
- References: uncontrolled/free dynamics and a tuned heuristic control.

The central alignment statistic is

$$
\operatorname{Disp}_v(t)=\frac{1}{N}\sum_{i=1}^N |v_t^i-\bar v_t|^2.
$$

The training objective is a particle approximation of

$$
J(\theta)=\mathbb{E}\left[\sum_{t=0}^{T-1} c(X_t,u_t,\mu_t)+g(X_T,\mu_T)\right],
$$

so lower is better.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from mfc.experiments import notebook_helpers as nh

ENV_NAME = "cucker-smale"
BASE_DIR = ROOT / "runs" / "extended_benchmark_bundles" / ENV_NAME
PRESET = "mid"
QUICK = PRESET == "smoke"
RUN_MISSING = False
FORCE_REBUILD = False
EXTENDED = True


In [ ]:
bundle = (
    nh.ensure_continuous_benchmark_bundle(
        ENV_NAME,
        BASE_DIR,
        quick=QUICK,
        force=FORCE_REBUILD,
        extended=EXTENDED,
        preset=PRESET,
    )
    if RUN_MISSING
    else nh.continuous_bundle_paths(ENV_NAME, BASE_DIR)
)

histories = nh.load_training_histories(bundle)
application = nh.load_application_data(bundle)
studies = nh.load_study_data(bundle)
grid_metrics = nh.load_study_grid_metrics(bundle)
diagnostics = nh.load_diagnostic_data(bundle)
optimization_history = nh.load_optimization_history(bundle)

bundle


## Main Results

These are the main figures for the nonlinear particle benchmark.

1. **Validation cost over training**: $J(\theta_k)$ with free and heuristic baselines.
2. **Population dynamics**: controlled trajectory compared with free and heuristic dynamics.
3. **Particle snapshots**: qualitative state of the controlled population at representative times.
4. **Control energy**: cumulative control effort used by the learned policy.


In [ ]:
display(nh.pathwise_main_summary_table(ENV_NAME, application))
nh.plot_pathwise_main_results(ENV_NAME, histories, application)


## Estimator Diagnostic Appendix

These panels summarize the continuous estimator chain:

$$
\mathbb{E}[d(M^\lambda,\mu)],
\qquad
\operatorname{MSE}(\widehat g),
\qquad
\operatorname{tr}\operatorname{Cov}(S_\lambda),
\qquad
\mathbb{E}\|\widehat D_t-D_t\|_2^2.
$$

For these particle systems, oracle diagnostics use the pathwise particle-gradient reference configured in the benchmark bundle.


In [ ]:
nh.plot_continuous_diagnostic_appendix(ENV_NAME, diagnostics)


## Optional Extended Studies

Run this section for budget allocation, horizon scaling, particle approximation, transfer, adaptive lambda, and ablations.


In [ ]:
nh.plot_budget_and_horizon(studies)
nh.plot_budget_pareto(studies, grid_metrics)
nh.plot_lambda_training_comparison(studies)
nh.plot_optimization_history(optimization_history)
nh.plot_optimization_summary(studies)
nh.plot_extended_study_summaries(studies, grid_metrics)


## Figure Coverage Audit

These tables map the broader requested figure list to saved artifacts. They are useful for checking completeness; they are not meant to be the headline story of the benchmark.


In [ ]:
nh.figure_checklist(ENV_NAME)


In [ ]:
nh.figure_coverage_matrix(ENV_NAME)


## Raw Tables For Custom Figures

The first few rows below are the main saved tables used by the plots above. Use these as the entry point for custom figures without rerunning training.


In [ ]:
sample_algorithm = next(iter(application))
sample_diag_algorithm = next(iter(diagnostics))
(
    application[sample_algorithm]["time_metrics"].head(),
    diagnostics[sample_diag_algorithm].get("gradient", pd.DataFrame()).head(),
    studies.get("budget", pd.DataFrame()).head(),
)
